# 02 — EDA and Forecasting Validation

This notebook explores the demand data and defines a forecasting validation design.

The key point is that forecasting validation must preserve time order. A random train-test split would leak future information and produce misleading results.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import generate_synthetic_healthcare_data
from clinic_forecast.validation import rolling_origin_windows, split_window

data_path = PROJECT_ROOT / "data" / "raw" / "clinic_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage, _, _ = generate_synthetic_healthcare_data()
    usage["date"] = pd.to_datetime(usage["date"])

usage.head()


## Seasonality diagnostics

Clinic demand has a strong weekday profile. This matters because staffing rules are usually planned by day of week.


In [ ]:
dow = (
    usage.groupby("day_of_week", as_index=False)["visits"]
    .mean()
    .assign(day_name=lambda x: x["day_of_week"].map({0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}))
)

fig, ax = plt.subplots()
ax.bar(dow["day_name"], dow["visits"])
ax.set_title("Average visits by day of week")
ax.set_xlabel("Day")
ax.set_ylabel("Average visits")
plt.show()

dow


In [ ]:
monthly = usage.groupby("month", as_index=False)["visits"].mean()

fig, ax = plt.subplots()
ax.plot(monthly["month"], monthly["visits"], marker="o")
ax.set_title("Average visits by month")
ax.set_xlabel("Month")
ax.set_ylabel("Average visits")
plt.show()


## Marketing exposure

Marketing spend is treated as an exogenous demand driver. In a real project, this could include campaigns, channel spend, referral partnerships and local outreach.


In [ ]:
campaign_effect = usage.groupby("campaign_active")["visits"].agg(["mean", "median", "count"])
campaign_effect


## Rolling-origin backtesting

The validation windows below simulate repeated forecasting dates. Each window trains on historical data and tests on the next 28 days.


In [ ]:
windows = list(
    rolling_origin_windows(
        usage,
        horizon_days=28,
        n_windows=4,
        min_train_days=365,
    )
)
pd.DataFrame([window.__dict__ for window in windows])


In [ ]:
train, test = split_window(usage, windows[-1])
print(train["date"].min(), train["date"].max(), train.shape)
print(test["date"].min(), test["date"].max(), test.shape)
